In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.interpolate import griddata
from scipy.spatial import KDTree
import figurefirst as fifi
import fly_plot_lib.plot as fpl
import os

import matplotlib as mpl

from splitflow import WindDataProcessing

Using device: cpu


In [ ]:
# Helvetica matches the published figure; log-axis tick labels use Computer Modern math text.
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['text.usetex'] = False
plt.rcParams["ps.usedistiller"] = 'xpdf'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams["mathtext.fontset"] = 'cm'

oranges = plt.cm.Oranges(np.linspace(0, 1, 10))
blues = plt.cm.Blues(np.linspace(0, 1, 10))
color_blue = oranges[6]
color_orange = blues[6]

In [ ]:
# Adds the CFD panels to the figure started by wind_tunnel_plot_maker.ipynb
# (run that notebook first). If it has not been run, start from the template.
import shutil
TEMPLATE_SVG = 'tunnel_methods_supplement_template.svg'
FIGURE_SVG = 'tunnel_methods_supplement.svg'
if not os.path.exists(FIGURE_SVG):
    shutil.copy(TEMPLATE_SVG, FIGURE_SVG)
layout = fifi.svg_to_axes.FigureLayout(FIGURE_SVG, autogenlayers=True, make_mplfigures=True, hide_layers=[])

# Make colorbars separately 

In [4]:
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    '''
    https://stackoverflow.com/a/18926541
    '''
    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)
    new_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap




In [ ]:
cmap_base = 'bwr'
vmin, vmax = 0.5, 1
Spectral_half_cmap = truncate_colormap(cmap_base, vmin, vmax)

# Panel O (vector strength) and panel P (instantaneous vorticity) colorbars,
# tick labels above the bar as in the published figure.
ax = layout.axes[('ColorBars', 'cb1')]
sm = mpl.cm.ScalarMappable(cmap='viridis', norm=mpl.colors.Normalize(vmin=0, vmax=1))
layout.figures['ColorBars'].colorbar(sm, cax=ax, orientation='horizontal')
fifi.mpl_functions.adjust_spines(ax, ['top'], xticks=[0, 1], spine_locations={'top': 0}, tick_length=1.5, linewidth=.5)
ax.set_xticklabels(['0', '1'])
ax.tick_params(pad=1.5)
fifi.mpl_functions.set_fontsize(ax, 6)

ax = layout.axes[('ColorBars', 'cb3')]
sm = mpl.cm.ScalarMappable(cmap=cmap_base, norm=mpl.colors.Normalize(vmin=-50, vmax=50))
layout.figures['ColorBars'].colorbar(sm, cax=ax, orientation='horizontal')
fifi.mpl_functions.adjust_spines(ax, ['top'], xticks=[-50, 0, 50], spine_locations={'top': 0}, tick_length=1.5, linewidth=.5)
ax.set_xticklabels(['−50', '0', '50'])
ax.tick_params(pad=1.5)
fifi.mpl_functions.set_fontsize(ax, 6)

layout.append_figure_to_layer(layout.figures['ColorBars'], 'ColorBars', cleartarget=True)

## Load in dfs

In [ ]:
df_XY_steady = WindDataProcessing.load_wind_data('../../../Data/2D_PlanarSlices/XY_steady/','csv', coord_system='CFD' )

In [ ]:
df_XY_unsteady = WindDataProcessing.load_wind_data('../../../Data/2D_PlanarSlices/XY_unsteady','csv',coord_system='CFD' )

In [ ]:
df_XZ = WindDataProcessing.load_wind_data('../../../Data/2D_PlanarSlices/XZslice','csv', coord_system='CFD')

In [9]:
def chunks(lst, n):
    arrays = []
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        arrays.append(lst[i:i + n])
    return arrays    

# XY Steady 

## Vector Strength

In [10]:
newdf = df_XY_steady.groupby(['x','y'], as_index=False,group_keys=False).apply(lambda d: 1/len(d)*np.sqrt((np.sum(d['unit_xvel'])**2) + (np.sum(d['unit_yvel'])**2)+(np.sum(d['unit_zvel'])**2)))



In [11]:
newdf.columns = ['x','y','vs']

### adding vector fields

In [12]:
XY_steady_average = df_XY_steady.groupby(['x','y'],as_index=False).mean()

In [13]:

# Create a grid of linearly spaced points within the range of x and y
x_linspace = np.linspace(XY_steady_average['x'].min()+.02, XY_steady_average['x'].max()-.02, 10)  # 5 points in x
y_linspace = np.linspace(XY_steady_average['y'].min()+.02, XY_steady_average['y'].max()-.02, 10)  # 5 points in y
X, Y = np.meshgrid(x_linspace, y_linspace)

# Flatten the grid and create a DataFrame
grid_points = pd.DataFrame({
    'x': X.flatten(),
    'y': Y.flatten()
})


# Create a KDTree from the original DataFrame points
tree = KDTree(XY_steady_average[['x', 'y']].values)

# Query the nearest points from the tree
distances, indices = tree.query(grid_points)

# Get the corresponding u and v values from the original DataFrame
nearest_points = XY_steady_average.iloc[indices]


In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf['x']),
    np.unique(newdf['y']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf['x'], newdf['y']),  # Original data points
    newdf['vs'],        # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

# Plotting
fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('E', 'e')] 
mesh=ax.pcolormesh(grid_x, grid_y, grid_z,vmin=0, vmax=1, shading='auto', cmap='viridis')

#plt.xlabel('X')
#plt.ylabel('Y')
#plt.title('Velocity Vector Strength')
#cbar = plt.colorbar(mesh, location='top')

#ax.axis('off')
ax.set_xticks([])
ax.set_yticks([])
ax.quiver(nearest_points['x'], nearest_points['y'], nearest_points['xvel'], nearest_points['yvel'], angles='xy', scale_units='xy',scale=10, color='grey')

#fig.set_edgecolor('darkturquoise')  
#fig.set_linewidth(10)  


for spine in ax.spines.values():
    spine.set_edgecolor('#e95e0dff')#('#57c0e1ff') #((87,192,225,100) )
    spine.set_linewidth(1.5)  

ax.set_rasterized(True)
ax.set_aspect('auto')  # fill the layout box, as in the published figure
fifi.mpl_functions.set_fontsize(ax, 6)
layout.append_figure_to_layer(layout.figures['E'], 'E', cleartarget=True)

# vorticity

## Instantaneous vorticity

In [18]:
newdf1 = df_XY_steady[df_XY_steady['Time']==25]

In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf1['x']),
    np.unique(newdf1['y']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf1['x'], newdf1['y']),  # Original data points
    newdf1['vorticity_z'],         # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

# Plotting
fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('G', 'g')]
mesh=ax.pcolormesh(grid_x, grid_y, grid_z, vmin=-50, vmax=50, shading='auto', cmap=cmap_base)

ax.set_xticks([])
ax.set_yticks([])
ax.quiver(nearest_points['x'], nearest_points['y'], nearest_points['xvel'], nearest_points['yvel'], angles='xy', scale_units='xy', scale=10, color='grey')

fifi.mpl_functions.set_fontsize(ax, 6)


for spine in ax.spines.values():
    spine.set_edgecolor('#e95e0dff')# color_blue ('#57c0e1ff') #((87,192,225,100) )
    spine.set_linewidth(1.5)  
ax.set_aspect('auto')  # fill the layout box, as in the published figure
ax.set_rasterized(True)
layout.append_figure_to_layer(layout.figures['G'], 'G', cleartarget=True)

# XY Unsteady 

## Vector Strength

In [20]:
newdf = df_XY_unsteady.groupby(['x','y'], as_index=False,group_keys=False).apply(lambda d: 1/len(d)*np.sqrt((np.sum(d['unit_xvel'])**2) + (np.sum(d['unit_yvel'])**2)+(np.sum(d['unit_zvel'])**2)))



In [21]:
newdf.columns = ['x','y','vs']

### adding vector fields

In [22]:
XY_unsteady_average = df_XY_unsteady.groupby(['x','y'],as_index=False).mean()

In [23]:

# Create a grid of linearly spaced points within the range of x and y
x_linspace = np.linspace(XY_unsteady_average['x'].min()+.02, XY_unsteady_average['x'].max()-.02, 10)  # 5 points in x
y_linspace = np.linspace(XY_unsteady_average['y'].min()+.02, XY_unsteady_average['y'].max()-.02, 10)  # 5 points in y
X, Y = np.meshgrid(x_linspace, y_linspace)

# Flatten the grid and create a DataFrame
grid_points = pd.DataFrame({
    'x': X.flatten(),
    'y': Y.flatten()
})


# Create a KDTree from the original DataFrame points
tree = KDTree(XY_unsteady_average[['x', 'y']].values)

# Query the nearest points from the tree
distances, indices = tree.query(grid_points)

# Get the corresponding u and v values from the original DataFrame
nearest_points = XY_unsteady_average.iloc[indices]


In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf['x']),
    np.unique(newdf['y']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf['x'], newdf['y']),  # Original data points
    newdf['vs'],        # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

# Plotting
fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('H', 'h')]
mesh=ax.pcolormesh(grid_x, grid_y, grid_z, vmin=0, vmax=1, shading='auto', cmap='viridis')

#plt.xlabel('X')
#plt.ylabel('Y')
#plt.title('Velocity Vector Strength')
#cbar = plt.colorbar(mesh, location='top')



ax.quiver(nearest_points['x'], nearest_points['y'], nearest_points['xvel'], nearest_points['yvel'], angles='xy', scale_units='xy', scale=10, color='grey')

#ax.axis('off')

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#3787c0ff')#('#e1a36dff') #((87,192,225,100) )
    spine.set_linewidth(1.5)  


fifi.mpl_functions.set_fontsize(ax, 6)
ax.set_aspect('auto')  # fill the layout box, as in the published figure
ax.set_rasterized(True)
layout.append_figure_to_layer(layout.figures['H'], 'H', cleartarget=True)

# vorticity

## Instantaneous vorticity

In [28]:
newdf1 = df_XY_unsteady[df_XY_unsteady['Time']==25]

In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf1['x']),
    np.unique(newdf1['y']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf1['x'], newdf1['y']),  # Original data points
    newdf1['vorticity_z'],         # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

# Plotting
fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('J', 'j')]
mesh=ax.pcolormesh(grid_x, grid_y, grid_z, vmin=-50,vmax=50, shading='auto', cmap=cmap_base)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#3787c0ff')# color_orange ('#e1a36dff') #((87,192,225,100) )
    spine.set_linewidth(1.5)  


ax.quiver(nearest_points['x'], nearest_points['y'], nearest_points['xvel'], nearest_points['yvel'], angles='xy', scale_units='xy', scale=10, color='grey')
fifi.mpl_functions.set_fontsize(ax, 6)
#ax.xlabel('X')
#plt.ylabel('Y')
#plt.title('Instantaneous Y Vorticity')
#cbar = plt.colorbar(mesh, location='top')
ax.set_aspect('auto')  # fill the layout box, as in the published figure
#ax.axis('off')
ax.set_rasterized(True)
layout.append_figure_to_layer(layout.figures['J'], 'J', cleartarget=True)

# XZ plane

## Vector Strength

In [30]:
newdf = df_XZ.groupby(['x','z'], as_index=False,group_keys=False).apply(lambda d: 1/len(d)*np.sqrt((np.sum(d['unit_xvel'])**2) + (np.sum(d['unit_yvel'])**2)+(np.sum(d['unit_zvel'])**2)))



In [31]:
newdf.columns = ['x','z','vs']

### adding vector fields

In [32]:
XZ_average = df_XZ.groupby(['x','z'],as_index=False).mean()

In [33]:

# Create a grid of linearly spaced points within the range of x and y
x_linspace = np.linspace(XZ_average['x'].min()+.02, XZ_average['x'].max()-.02, 10)  # 5 points in x
y_linspace = np.linspace(XZ_average['z'].min()+.02, XZ_average['z'].max()-.02, 10)  # 5 points in y
X, Y = np.meshgrid(x_linspace, y_linspace)

# Flatten the grid and create a DataFrame
grid_points = pd.DataFrame({
    'x': X.flatten(),
    'z': Y.flatten()
})


# Create a KDTree from the original DataFrame points
tree = KDTree(XZ_average[['x', 'z']].values)

# Query the nearest points from the tree
distances, indices = tree.query(grid_points)

# Get the corresponding u and v values from the original DataFrame
nearest_points = XZ_average.iloc[indices]


In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf['x']),
    np.unique(newdf['z']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf['x'], newdf['z']),  # Original data points
    newdf['vs'],        # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

# Plotting
fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('K', 'k')]
mesh=ax.pcolormesh(grid_x, grid_y, grid_z, vmin=0, vmax=1, shading='auto', cmap='viridis')

#ax.set_xlabel('X')
#ax.set_ylabel('Z')
#ax.set_title('Velocity Vector Strength')
#fig.colorbar(mesh, location='top',aspect=40).outline.set_visible(False)

ax.quiver(nearest_points['x'], nearest_points['z'], nearest_points['xvel'], nearest_points['zvel'], angles='xy', scale_units='xy', scale=10, color='grey', alpha=.7)
#fifi.mpl_functions.adjust_spines(ax, ['left','bottom'], yticks =[0,0.5], xticks=[0,1.25],  tick_length=3, linewidth=.5)
#ax.axis('off')

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#dc2b94ff') #((87,192,225,100) )
    spine.set_linewidth(1.5)  


ax.set_aspect('auto')  # fill the layout box, as in the published figure
fifi.mpl_functions.set_fontsize(ax, 6)
ax.set_rasterized(True)
layout.append_figure_to_layer(layout.figures['K'], 'K', cleartarget=True)

# vorticity

## Instantaneous vorticity

In [37]:
newdf1 = df_XZ[df_XZ['Time']==25]

In [ ]:
# Prepare the grid
grid_x, grid_y = np.meshgrid(
    np.unique(newdf1['x']),
    np.unique(newdf1['z']),
)

# Interpolate to fill missing values
grid_z = griddata(
    (newdf1['x'], newdf1['z']),  # Original data points
    newdf1['vorticity_z'],         # Values at those points
    (grid_x, grid_y),   # Grid to interpolate onto
    method='linear'     # Interpolation method
)

fig, ax = plt.subplots(figsize=(9, 6),layout='constrained')
ax = layout.axes[('M', 'm')]
mesh=ax.pcolormesh(grid_x, grid_y, grid_z, vmin=-50,vmax=50, shading='auto', cmap=cmap_base)

#fig.colorbar(mesh, location='top',aspect=40).outline.set_visible(False)

ax.quiver(nearest_points['x'], nearest_points['z'], nearest_points['xvel'], nearest_points['zvel'], angles='xy', scale_units='xy', scale=10, color='grey')

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#dc2b94ff') #((87,192,225,100) ) #f747a5ff
    spine.set_linewidth(1.5)  


ax.set_aspect('auto')  # fill the layout box, as in the published figure

ax.set_rasterized(True)

fifi.mpl_functions.set_fontsize(ax, 6)

layout.append_figure_to_layer(layout.figures['M'], 'M', cleartarget=True)

## FFTs


### Austin's anemometer measurement locations: 

### For the horizontal spitter plate

steady:[0.5334, 0.2413, 0.1143], 
unsteady:[0.5334, 0.2413, 0.381],  
laminar: [0.5334, 0.2413, 0.2413]

In [39]:
df_loc_steady = df_XY_steady[(df_XY_steady.x==np.float32(0.81556)) & (df_XY_steady.y==np.float32(.21601))].copy()

In [40]:
df_loc_unsteady = df_XY_unsteady [(df_XY_unsteady.x==np.float32(0.81556)) & (df_XY_unsteady.y==np.float32(.21601))].copy()

In [41]:
df_loc_steady['3d_speed']= np.sqrt(df_loc_steady.xvel**2 + df_loc_steady.yvel**2 + df_loc_steady.zvel**2)
df_loc_unsteady['3d_speed']= np.sqrt(df_loc_unsteady.xvel**2 + df_loc_unsteady.yvel**2 + df_loc_unsteady.zvel**2)

In [42]:
df_loc_steady['2d_speed']= np.sqrt(df_loc_steady.xvel**2 + df_loc_steady.yvel**2 )
df_loc_unsteady['2d_speed']= np.sqrt(df_loc_unsteady.xvel**2 + df_loc_unsteady.yvel**2 )

In [43]:

freqs_s, ps_s, psd_s =WindDataProcessing.FFT(df_loc_steady['2d_speed'], dt=.1)

freqs_uns, ps_uns, psd_uns = WindDataProcessing.FFT(df_loc_unsteady['2d_speed'], dt=.1)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
ax = layout.axes[('A', 'a')]

ax.loglog(freqs_s, psd_s, linewidth=1, label='xy speed',color=color_blue,alpha=.95)

ax.loglog(freqs_uns, psd_uns, linewidth=1, label='xy speed',color=color_orange,alpha=.95)
ax.set_xlim([10**-2, 10**1])
ax.set_ylim([10**-8, 10**-1])
fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'], yticks =[10**-8, 10**-1], xticks=[10**-2, 10**1],
                                tick_length=3,spine_locations={'left': 5, 'bottom': 5}, linewidth=1)

# axis labels are text in the template
fifi.mpl_functions.set_fontsize(ax, 6)
ax.minorticks_off()

fifi.mpl_functions.set_fontsize(ax, 6)
layout.append_figure_to_layer(layout.figures['A'], 'A', cleartarget=True)

fig, ax = plt.subplots(figsize=(10,3))


plt.loglog(freqs_x, psd_x, label='x', )

plt.loglog(freqs_y, psd_y, label='y', )
plt.loglog(freqs_z, psd_z, label='z', )



plt.legend()
plt.xlabel('Frequency')
plt.ylabel('PSD')
plt.xlim(.01,10**1)


# Time Series Plots

In [45]:
df_XY_steady['D'] = np.arctan2(df_XY_steady['yvel'], df_XY_steady['xvel'])
df_XY_steady['s2'] = np.sqrt(df_XY_steady['yvel']**2 + df_XY_steady['xvel']**2)

df_XY_unsteady['D'] = np.arctan2(df_XY_unsteady['yvel'], df_XY_unsteady['xvel'])
df_XY_unsteady['s2'] =  np.sqrt(df_XY_unsteady['yvel']**2 + df_XY_unsteady['xvel']**2)

In [49]:
#middle_point_steady[['x','y','z','xvel','yvel','zvel','vorticity_x', 'vorticity_y','vorticity_z','Time']].to_hdf('../../../Data/Fig1_Input_Data/steady_timeseries.hdf', key='time')
#middle_point_unsteady[['x','y','z','xvel','yvel','zvel','vorticity_x', 'vorticity_y','vorticity_z','Time']].to_hdf('../../../Data/Fig1_Input_Data/unsteady_timeseries.hdf', key='time')




In [51]:
#inlet_x= .27
#outlet_x= 1.18

In [52]:
# take some extra points around the area for added measurements
middle_points_steady = df_XY_steady[(df_XY_steady['x'].between(0.81,.83) )& (df_XY_steady['y'].between(.2,.24))].copy()

middle_points_unsteady = df_XY_unsteady[(df_XY_unsteady['x'].between(0.81,.83)  )& (df_XY_unsteady['y'].between(.2,.24))].copy()

In [90]:
middle_points_steady.x.max()-middle_points_steady.x.min()

0.018130004405975342

In [ ]:
# Create a new figure
fig = plt.figure()

# Add an axes to the figure, specifying it as a polar plot
ax = fig.add_axes([0, 0, 1, 1], polar=True)
ax = layout.axes[('B', 'b')]
ax.scatter(middle_points_steady['D'],middle_points_steady['s2'],s=.2, edgecolor=None, color=color_blue, alpha=1)
ax.scatter(middle_points_unsteady['D'],middle_points_unsteady['s2'],s=.2,  edgecolor=None, color=color_orange, alpha=1)

ax.set_rmax(1.5)
ax.set_rticks([0.75,1])  # Less radial ticks

# ax.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/2, np.pi , -3*np.pi/2, -np.pi/2, -np.pi/4])
#ax.set_xticklabels(['0', '$-\pi/4$', '$-\pi/2$', '$-3\pi/4$', '$\pm\pi$', '$3\pi/4$', '$\pi/2$', '$\pi/4$'])
ax.set_xticklabels([])

# Set the yticks off
ax.set_yticks([.75,1.5])
ax.set_yticklabels([])  # radial labels are text in the template

# Set the direction of the plot to clockwise
ax.set_rlabel_position(180)
ax.set_theta_direction(-1)

# Set the zero to the top
ax.set_theta_offset(0)

fifi.mpl_functions.set_fontsize(ax, 6)



layout.append_figure_to_layer(layout.figures['B'], 'B', cleartarget=True)

In [ ]:
fig,ax = plt.subplots(figsize=(1,3))
ax = layout.axes[('C', 'c')]

feature_val = middle_points_steady.zvel.values # df.groupby('obj_id_unique')[feature_name].mean().values 

b, h = np.histogram(feature_val)
fpl.scatter_box_violin_distribution(ax, 0, feature_val, b, h,markersize=1, shading=1, edgecolor=None, linewidth=None, color='#ff7f0e')  # steady


feature_val = middle_points_unsteady.zvel.values # df.groupby('obj_id_unique')[feature_name].mean().values 

b, h = np.histogram(feature_val)
#,markersize=0.2, edgecolor=None, 

fpl.scatter_box_violin_distribution(ax, 1, feature_val, b, h,markersize=1, shading=1, edgecolor=None, linewidth=None, color='#3787c0')  # unsteady

#ax, x, y_data, hist_counts, hist_bins,


fifi.mpl_functions.adjust_spines(ax, ['left'], yticks =[-1,-.5,0,.5,1], spine_locations={'left': 2}, tick_length=3, linewidth=1)
ax.set_yticklabels([-1,-.5,0,.5,1])

# y label is text in the template



fifi.mpl_functions.set_fontsize(ax, 6)
layout.append_figure_to_layer(layout.figures['C'], 'C', cleartarget=True)

In [56]:
# take some extra points around the area for added measurements
middle_points_steady = df_XY_steady[(df_XY_steady['x'].between(0.81,.83) )& (df_XY_steady['y'].between(.2,.24))].copy()

middle_points_unsteady = df_XY_unsteady[(df_XY_unsteady['x'].between(0.81,.83)  )& (df_XY_unsteady['y'].between(.2,.24))].copy()

fig,ax = plt.subplots(figsize=(4,4))

#ax = layout.axes[('D', 'd')]

ax.scatter(ti_v_dir_steadydf.dir_std, ti_v_dir_steadydf.ti, s=20,  color=color_blue)
ax.scatter(ti_v_dir_unsteadydf.dir_std, ti_v_dir_unsteadydf.ti,  s=20,  color=color_orange)



ax.set_xlim(0,90)
ax.set_ylim(0,.8)


fifi.mpl_functions.adjust_spines(ax, ['left','bottom'], yticks =[0,.2,.4,.6,0.8], xticks=[0,30,60,90], spine_locations={'left': 5, 'bottom': 5}, tick_length=3, linewidth=1)

ax.set_ylabel('Turbulence Intensity')
ax.set_xlabel('Standard Deviation in \n Direction ($^\circ$)')


fifi.mpl_functions.set_fontsize(ax, 6)

#layout.append_figure_to_layer(layout.figures['D'], 'D', cleartarget=True)


# Plume snapshots

In [ ]:
layout.write_svg(FIGURE_SVG)